###  **1. Importing Required Libraries**

In [ ]:
!pip install umap-learn

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from umap import UMAP, AlignedUMAP
from mane import mane_2set

###  **2. Applying Dimensionality Reduction**

####  **2.1 Computing Embeddings**

In [17]:
def compute_umap_embeddings(modality_1, modality_2):
    """
    Compute UMAP embeddings for modality 1, modality 2, and aligned UMAP embedding, and MANE.

    Args:
        modality_1 (np.ndarray): Data of modality 1, N instances with dimension n, shape (N, n).
        modality_2 (np.ndarray): Data of modality 2, N instances with dimension n, shape (N, n).

    Returns:
        umap_modality_1 (np.ndarray): 2D UMAP embedding of modality 1 data.
        umap_modality_2 (np.ndarray): 2D UMAP embedding of modality 2 data.
        aligned_embedding (list of np.ndarray): List containing aligned UMAP embeddings [modality 1, modality 2].
        mane_embedding (np.ndarray): Mane embedding of modality 1 and modality 2.
    """

    # Concatenate signals internally
    concat = np.concatenate([modality_1, modality_2], axis=1)  # shape (num_beats, 512)

    # Define UMAP parameters for individual datasets
    ump = UMAP(n_components=2, min_dist=0.1, n_neighbors=15, random_state=42, init='pca')

    # UMAP on modality 1
    umap_modality_1 = ump.fit_transform(modality_1)

    # UMAP on modality 2
    umap_modality_2 = ump.fit_transform(modality_2)

    # Define relations for aligned UMAP (one-to-one beat correspondence)
    relations = [{i: i for i in range(modality_1.shape[0])}]

    # Aligned UMAP embedding
    aligned_ump = AlignedUMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        alignment_window_size=2,
        random_state=42
    )
    aligned_embedding = aligned_ump.fit_transform([modality_1, modality_2], relations=relations)

    mane_embedding, _, _ = mane_2set(modality_1, modality_2)

    return umap_modality_1, umap_modality_2, aligned_embedding, mane_embedding


In [ ]:
# =============================================================================
# Embedding Visualization and KNN Classification for Multi-Modal Cell Imaging
# Dataset: Individual cells imaged across multiple modalities (Brightfield & Phase)
# =============================================================================

# load your data
# data = ...

# For our case, as written in the paper: 
X_selected = data["X_selected"]            # shape (N, 2, 64, 64, 3)
modality1 = X_selected[:, 0, :, :, :]      # Brightfield images, shape (N, 64, 64, 3)
modality2 = X_selected[:, 1, :, :, :]      # Phase images, shape (N, 64, 64, 3)

# --- Vectorize each modality: flatten spatial dims (64x64x3 = 12288) per cell ---
N = modality1.shape[0]
modality1_vec = modality1.reshape(N, -1)    # shape: (N, 12288) — one vector per cell, Brightfield
modality2_vec = modality2.reshape(N, -1)    # shape: (N, 12288) — one vector per cell, Phase

# === COMPUTE EMBEDDINGS ===
# Each modality is embedded separately (UMAP), then jointly aligned (Joint UMAP),
# and finally fused via manifold alignment (MANE) for cross-modal comparison
umap1, umap2, aligned_emb, mane_embedding = compute_umap_embeddings(modality1_vec, modality2_vec)

umap_embeddings   = [umap1, umap2]          # Per-modality UMAP embeddings
aligned_embeddings = [aligned_emb]           # Joint UMAP embedding (modality-aligned)
ground_truth = data["ground_truth"]          # Cell viability labels: 0 = Live, 1 = Dead, shape (N,)

if ground_truth is None:
    raise ValueError("Ground truth labels are required for this plot, but none were found in data.")

modality_names = ["Brightfield", "Phase"]

print("UMAP embeddings shape:   ", np.shape(umap_embeddings))
print("MANE embedding shape:    ", np.shape(mane_embedding))
print("Aligned embeddings shape:", [np.shape(emb) for emb in aligned_embeddings])


####  **2.2 Visualization and KNN**

In [ ]:
# === KNN CLASSIFICATION ===
# Evaluate how well each embedding separates Live vs Dead cells
# using K-Nearest Neighbors at multiple values of K
k_values = [5, 10, 15, 20, 30, 50, 80, 100, 120, 150, 180, 200]
accuracy_results = []

knn_colors = {
    f"UMAP {modality_names[0]}": "red",
    f"UMAP {modality_names[1]}": "blue",
    "Joint UMAP"               : "orange",
    "MANE"                     : "green"
}

# Each entry: (embedding array, display name)
embeddings_list = [
    (umap_embeddings[0],    f"UMAP {modality_names[0]}"),  # Brightfield only
    (umap_embeddings[1],    f"UMAP {modality_names[1]}"),  # Phase only
    (aligned_embeddings[0], "Joint UMAP"),                  # Jointly aligned across modalities
    (mane_embedding,        "MANE"),                        # Manifold-aligned multi-modal fusion
]

for embedding, name in embeddings_list:
    embedding_acc = []
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(embedding, ground_truth)
        y_pred = knn.predict(embedding)
        accuracy = accuracy_score(ground_truth, y_pred) * 100
        embedding_acc.append(accuracy)
    accuracy_results.append((name, embedding_acc))

# === PLOT: Embedding Scatter Plots + KNN Accuracy Curve ===
# Layout: [UMAP BF | UMAP Phase | Joint UMAP | MANE | KNN Accuracy vs K]
fig, axes = plt.subplots(1, 5, figsize=(40, 7))
fontsize        = 28
legend_fontsize = 24
label_fontsize  = 20
point_size      = 1

legend_elements = [
    Patch(facecolor="green", edgecolor="green", label="Live"),
    Patch(facecolor="red",   edgecolor="red",   label="Dead")
]

# --- Panel 1: UMAP Brightfield ---
ax  = axes[0]
emb = umap_embeddings[0]
ax.scatter(emb[ground_truth == 0, 0], emb[ground_truth == 0, 1], s=point_size, color="green", alpha=0.7)
ax.scatter(emb[ground_truth == 1, 0], emb[ground_truth == 1, 1], s=point_size, color="red",   alpha=0.7)
ax.set_title("UMAP Embedding of Brightfield", fontsize=fontsize)
ax.set_xlabel("UMAP 1", fontsize=label_fontsize)
ax.set_ylabel("UMAP 2", fontsize=label_fontsize)
ax.legend(handles=legend_elements, frameon=False, fontsize=legend_fontsize)

# --- Panel 2: UMAP Phase (flip axis 0 for visual alignment with Brightfield) ---
ax  = axes[1]
emb = umap_embeddings[1].copy()
emb[:, 0] = -emb[:, 0]
ax.scatter(emb[ground_truth == 0, 0], emb[ground_truth == 0, 1], s=point_size, color="green", alpha=0.7)
ax.scatter(emb[ground_truth == 1, 0], emb[ground_truth == 1, 1], s=point_size, color="red",   alpha=0.7)
ax.set_title("UMAP Embedding of Phase", fontsize=fontsize)
ax.set_xlabel("UMAP 1", fontsize=label_fontsize)
ax.set_ylabel("UMAP 2", fontsize=label_fontsize)
ax.legend(handles=legend_elements, frameon=False, fontsize=legend_fontsize)

# --- Panel 3: Joint UMAP (modalities aligned in a shared embedding space) ---
ax  = axes[2]
emb = aligned_embeddings[0]
ax.scatter(emb[ground_truth == 0, 0], emb[ground_truth == 0, 1], s=point_size, color="green", alpha=0.7)
ax.scatter(emb[ground_truth == 1, 0], emb[ground_truth == 1, 1], s=point_size, color="red",   alpha=0.7)
ax.set_title("Joint UMAP Embedding", fontsize=fontsize)
ax.set_xlabel("UMAP 1", fontsize=label_fontsize)
ax.set_ylabel("UMAP 2", fontsize=label_fontsize)
ax.legend(handles=legend_elements, frameon=False, fontsize=legend_fontsize)

# --- Panel 4: MANE (flip axis 0 for visual alignment) ---
ax  = axes[3]
emb = mane_embedding.copy()
# emb[:, 0] = -emb[:, 0]
ax.scatter(emb[ground_truth == 0, 0], emb[ground_truth == 0, 1], s=point_size, color="green", alpha=0.7)
ax.scatter(emb[ground_truth == 1, 0], emb[ground_truth == 1, 1], s=point_size, color="red",   alpha=0.7)
ax.set_title("MANE Embedding", fontsize=fontsize)
ax.set_xlabel("MANE 1", fontsize=label_fontsize)
ax.set_ylabel("MANE 2", fontsize=label_fontsize)
ax.legend(handles=legend_elements, frameon=False, fontsize=legend_fontsize)

# --- Panel 5: KNN Accuracy vs K — compares separability across all embeddings ---
ax = axes[4]
for name, acc in accuracy_results:
    color = knn_colors.get(name, "black")
    ax.plot(k_values, acc, marker="o", label=name, linewidth=2, color=color)
ax.set_title("KNN Accuracy vs K", fontsize=fontsize)
ax.set_xlabel("K value", fontsize=label_fontsize)
ax.set_ylabel("Accuracy (%)", fontsize=label_fontsize)
ax.legend(fontsize=legend_fontsize - 4, loc="best")

plt.tight_layout()
plt.show()